# SituationEngine — QLoRA Fine-tune (Llama 3.1 8B, A100)

Colab driver for Phase 2. This notebook does **not** define training logic; it only configures and calls the orchestrator in `app.llm.training.situation_engine_finetune`. All defaults match the unbreakable rules in `docs/intelligence_migration.md`.

Expected runtime on a single A100 40 GB: ~45 min for 180 training scenarios at 3 epochs (sequence length 4096, micro-batch 1, gradient-accumulation 16).

## 1. Environment

In [ ]:
!nvidia-smi

In [ ]:
# Pin to versions known to work with Llama 3.1 8B QLoRA on A100.
%pip install --quiet \
    "transformers>=4.43,<4.46" \
    "peft>=0.11,<0.13" \
    "accelerate>=0.31" \
    "bitsandbytes>=0.43" \
    "datasets>=2.20" \
    "sentencepiece" \
    "pydantic>=2.5" \
    "pyyaml"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Repository checkout

In [ ]:
%cd /content
# Replace the URL below with your private mirror or use Drive checkout.
REPO_URL = 'https://example.invalid/Social-Media-Radar.git'
!git clone --depth 1 {REPO_URL} repo || true
%cd /content/repo
import sys; sys.path.insert(0, '/content/repo')

## 3. Authenticate to HuggingFace and pull data

The training JSONL is produced by `scripts/build_training_set.py` against the labelled corpus and should be staged on Drive (or a private bucket) before this notebook runs. Do **not** put raw labels in the repo.

In [ ]:
from huggingface_hub import login
login()  # paste an HF token that has access to Meta-Llama-3.1-8B-Instruct

In [ ]:
DATA_SRC = '/content/drive/MyDrive/situation_engine/training'
!mkdir -p data/training
!cp {DATA_SRC}/train.jsonl data/training/train.jsonl
!cp {DATA_SRC}/val.jsonl data/training/val.jsonl
!wc -l data/training/*.jsonl

## 4. Pre-flight (CPU)

In [ ]:
from pathlib import Path
from app.llm.training.situation_engine_finetune import (
    SituationEngineFineTuneConfig, SituationEngineFineTuner,
)

OUT = Path('/content/drive/MyDrive/situation_engine/checkpoints/run_001')
OUT.mkdir(parents=True, exist_ok=True)

cfg = SituationEngineFineTuneConfig(
    train_file=Path('data/training/train.jsonl'),
    val_file=Path('data/training/val.jsonl'),
    output_dir=OUT,
)
ft = SituationEngineFineTuner(cfg)
ft.preflight()

## 5. Train

In [ ]:
metrics = ft.run()
metrics

## 6. Promote (held-out judge gate)

In [ ]:
import os, subprocess
os.environ.setdefault('OPENAI_API_KEY', 'paste-here-or-set-in-Colab-secrets')
os.environ.setdefault('ANTHROPIC_API_KEY', 'paste-here-or-set-in-Colab-secrets')
rc = subprocess.call([
    'python', 'scripts/promote_checkpoint.py',
    '--base-model', cfg.base_model,
    '--lora-weights', str(OUT / 'final'),
    '--scenarios-root', 'data/scenarios',
    '--floor', '85',
    '--manifest', str(OUT / 'promotion_manifest.json'),
])
print('promotion exit code:', rc)